
Graph-State Encrypted Cloning Certification (GSECC)
and Graph-State Decoder Construction (GSDC)
===========================================================================

This module implements the complete computational pipeline for the exact
certification of graph-state encrypted cloning resources and the
construction of the corresponding decoder.

The implementation follows the algorithms described in the accompanying
README and consists of six logical components.

---------------------------------------------------------------------------
1. GF(2) LINEAR ALGEBRA
---------------------------------------------------------------------------

Functions

    gf2_rank()
    gf2_is_invertible()

These routines implement Gaussian elimination over the binary field GF(2)
and are used to determine the rank and invertibility of the cut matrix

$$
\Gamma_{S,N}.
$$

The rank computation is the computational core of the GSECC certification
algorithm.

---------------------------------------------------------------------------
2. GRAPH UTILITIES
---------------------------------------------------------------------------

Functions

    adjacency_matrix()
    cut_matrix()

These utilities construct graph adjacency matrices and extract the cut
matrix associated with a balanced bipartition

$$
V=S\cup N.
$$

---------------------------------------------------------------------------
3. PARAMETER VALIDATION
---------------------------------------------------------------------------

Functions

    validate_graph_size()
    print_parameters()

These routines verify that the supplied graph satisfies

$$
|V|=2mk,
$$

where

- m : number of clones,
- k : number of signal qubits,
- mk : subsystem size.

Invalid parameter choices immediately raise a ValueError.

---------------------------------------------------------------------------
4. GSECC
---------------------------------------------------------------------------

Functions

    verify_certificate()
    find_certificate_exact()

verify_certificate()

    Polynomial-time verification of a candidate certificate.

    Returns True iff

   $$
\text{rank}_{GF(2)}(\Gamma_{S,N}) = mk.
$$

find_certificate_exact()

    Implements the exact certification algorithm.

    The routine

        • enumerates every balanced partition,
        • constructs Γ(S,N),
        • computes its GF(2) rank,
        • returns the first valid certificate.

    Returns

        (S,N)

    if a certificate exists.

    Returns

        None

    only when every balanced partition has been exhausted.

    Therefore, None constitutes a rigorous proof that no valid
    encrypted-cloning partition exists.

---------------------------------------------------------------------------
5. GRAPH-STATE CONSTRUCTION
---------------------------------------------------------------------------

Functions

    graph_state_from_adjacency()
    ghz_state()

    bell_pair_matching_graph()
    star_graph()
    path_graph()
    grid_graph()

These routines generate graph states and benchmark graph families used
throughout the demonstrations.

---------------------------------------------------------------------------
6. REDUCED DENSITY MATRICES
---------------------------------------------------------------------------

Functions

    reduced_density_matrix()
    rho_S_matches_maximally_mixed()

These routines compute

$$
\rho_S
$$

and verify whether

$$
\rho_S=\frac{I}{2^{mk}}
$$

within numerical precision.

Only the residual norm is reported.

The density matrix itself is not printed.

---------------------------------------------------------------------------
7. GSDC
---------------------------------------------------------------------------

Functions

    construct_decoder()

Given a certified partition

$$
(S,N),
$$

constructs the decoder unitary

$$
W
$$

satisfying

$$
|G\rangle
=
(I_S\otimes W)
|\Phi_{2^{mk}}\rangle.
$$

The routine additionally verifies

    • unitarity,

    • state reconstruction,

    • numerical residuals.

---------------------------------------------------------------------------
8. VISUALIZATION
---------------------------------------------------------------------------

Function

    plot_graph()

Generates publication-quality graph visualizations.

Certified graphs display

    • signal subsystem,

    • noise subsystem,

    • cut edges,

using separate colours.

Figures are automatically saved as

    • PDF

    • 600 dpi PNG

---------------------------------------------------------------------------
9. REPORTING
---------------------------------------------------------------------------

Functions

    report_certificate()

Runs the complete certification pipeline

    GF(2) verification
        ↓
    reduced density matrix
        ↓
    decoder construction
        ↓
    graph visualization

report_non_graph_state()

Applies the GSDC verification pipeline to arbitrary quantum states such
as GHZ states.

---------------------------------------------------------------------------
10. DEMONSTRATION
---------------------------------------------------------------------------

Executing this file automatically evaluates

    • Complete graphs
    • Cycle graphs
    • Linear cluster graphs
    • Cluster grids
    • Bell-pair matching graphs
    • GHZ graphs
    • Dense Erdős–Rényi graphs
    • Sparse Erdős–Rényi graphs
    • GHZ state vectors
    • Invalid (m,k) examples

For every certified graph the script automatically

    • identifies the certificate,

    • verifies maximal mixedness,

    • constructs the decoder,

    • generates publication-quality figures,

    • reports numerical residuals.

For the complete mathematical formulation, proofs, computational
complexity, and theoretical background, see README.md.


In [1]:

from __future__ import annotations
import itertools
import os
import random
from typing import Iterable, Optional, Sequence, Tuple

import numpy as np

import matplotlib
matplotlib.use("Agg")  # headless / script-safe backend, no plt.show() needed
import matplotlib.pyplot as plt
import networkx as nx

try:
    FIGURE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    FIGURE_DIR = os.getcwd()



In [2]:
# GF(2) linear algebra

def gf2_rank(mat: np.ndarray) -> int:
    A = mat.copy().astype(np.uint8) % 2
    rows, cols = A.shape
    rank = 0
    for col in range(cols):
        pivot = None
        for r in range(rank, rows):
            if A[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        A[[rank, pivot]] = A[[pivot, rank]]
        for r in range(rows):
            if r != rank and A[r, col]:
                A[r, :] ^= A[rank, :]
        rank += 1
        if rank == rows:
            break
    return rank


def gf2_is_invertible(mat: np.ndarray) -> bool:
    n, m = mat.shape
    if n != m:
        raise ValueError("Matrix must be square to test invertibility.")
    return gf2_rank(mat) == n


# Graph representation & cut matrix

def adjacency_matrix(n: int, edges: Iterable[Tuple[int, int]]) -> np.ndarray:
    A = np.zeros((n, n), dtype=np.uint8)
    for u, v in edges:
        A[u, v] = 1
        A[v, u] = 1
    return A


def cut_matrix(A: np.ndarray, S: Sequence[int], N: Sequence[int]) -> np.ndarray:
    return A[np.ix_(S, N)]



In [3]:
# (m, k) parameter validation

def validate_graph_size(A: np.ndarray, m: int, k: int) -> int:
    if m <= 0 or k <= 0:
        raise ValueError(f"m and k must be positive integers (got m={m}, k={k}).")
    mk = m * k
    n = A.shape[0]
    if n != 2 * mk:
        raise ValueError(
            f"Graph has {n} vertices, but expected 2*m*k = {2 * mk} "
            f"for m={m}, k={k}."
        )
    return mk


def print_parameters(m: int, k: int, A: np.ndarray) -> None:
    mk = m * k
    n = A.shape[0]
    print("Encrypted Cloning Parameters")
    print("----------------------------")
    print(f"m  = {m}")
    print(f"k  = {k}")
    print(f"mk = {mk}")
    print(f"Graph vertices = {n}")


# GSECC: verification of a single candidate certificate (S, N)

def verify_certificate(A: np.ndarray, S: Sequence[int], N: Sequence[int], m: int, k: int) -> bool:
    mk = validate_graph_size(A, m, k)
    n = A.shape[0]
    if len(S) != len(N) or len(S) + len(N) != n or len(S) != mk:
        raise ValueError(f"S, N must each have size mk = {mk} and partition all {n} vertices.")
    if set(S) & set(N):
        raise ValueError("S and N must be disjoint.")
    Gamma = cut_matrix(A, list(S), list(N))
    return gf2_is_invertible(Gamma)


# GSECC: exhaustive search (exact, exponential -- fine for small n)

def find_certificate_exact(A: np.ndarray, m: int, k: int) -> Optional[Tuple[Tuple[int, ...], Tuple[int, ...]]]:
    mk = validate_graph_size(A, m, k)
    n = A.shape[0]
    vertices = range(n)
    others = [v for v in vertices if v != 0]
    for combo in itertools.combinations(others, mk - 1):
        S = (0,) + combo
        N = tuple(v for v in vertices if v not in S)
        if verify_certificate(A, S, N, m, k):
            return S, N
    return None


In [4]:

# Exact graph-state / GHZ / Bell-pair construction

def graph_state_from_adjacency(A: np.ndarray) -> np.ndarray:
    n = A.shape[0]
    plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
    psi = plus
    for _ in range(n - 1):
        psi = np.kron(psi, plus)
    psi = psi.reshape([2] * n)
    for i in range(n):
        for j in range(i + 1, n):
            if A[i, j]:
                for bits in itertools.product([0, 1], repeat=n):
                    if bits[i] == 1 and bits[j] == 1:
                        psi[bits] *= -1
    return psi.reshape(-1)


def ghz_state(n: int) -> np.ndarray:
    psi = np.zeros(2 ** n, dtype=complex)
    psi[0] = 1 / np.sqrt(2)
    psi[-1] = 1 / np.sqrt(2)
    return psi


def bell_pair_matching_graph(m: int, k: int) -> np.ndarray:
    mk = m * k
    edges = [(i, i + mk) for i in range(mk)]
    return adjacency_matrix(2 * mk, edges)


def star_graph(m: int, k: int) -> np.ndarray:
    mk = m * k
    n = 2 * mk
    edges = [(0, v) for v in range(1, n)]
    return adjacency_matrix(n, edges)


def path_graph(m: int, k: int) -> np.ndarray:
    mk = m * k
    n = 2 * mk
    edges = [(i, i + 1) for i in range(n - 1)]
    return adjacency_matrix(n, edges)


def grid_graph(rows: int, cols: int) -> np.ndarray:
    n = rows * cols
    edges = []
    for r in range(rows):
        for c in range(cols):
            v = r * cols + c
            if c + 1 < cols:
                edges.append((v, v + 1))
            if r + 1 < rows:
                edges.append((v, v + cols))
    return adjacency_matrix(n, edges)


def reduced_density_matrix(state: np.ndarray, keep: Sequence[int], n: int) -> np.ndarray:
    keep = list(sorted(keep))
    trace = [i for i in range(n) if i not in keep]
    psi = state.reshape([2] * n)
    perm = keep + trace
    psi = np.transpose(psi, perm)
    dk = 2 ** len(keep)
    dt = 2 ** len(trace)
    psi = psi.reshape(dk, dt)
    return psi @ psi.conj().T


def rho_S_matches_maximally_mixed(rho_S: np.ndarray, m: int, k: int, atol: float = 1e-9):
    mk = m * k
    target = np.eye(2 ** mk) / (2 ** mk)
    residual = np.linalg.norm(rho_S - target)
    return residual <= atol, residual


In [5]:

# Visualization

def plot_graph(A: np.ndarray, S: Optional[Sequence[int]] = None, N: Optional[Sequence[int]] = None, title: str = "Graph", save: bool = True, fig_dir: str = FIGURE_DIR,) -> Optional[str]:
    
    G = nx.from_numpy_array(A)

    plt.figure(figsize=(8, 7))

    if S is None or N is None:
        pos = nx.spring_layout(G, seed=5)
        nx.draw_networkx(
            G,
            pos,
            node_color="#4CAF50",
            node_size=700,
            edgecolors="black",
            linewidths=1.2,
            font_size=12,
            font_weight="bold",
        )
    else:
        S = list(S)
        N = list(N)
        mk = len(S)

        pos = {}
        y = list(range(mk))[::-1]

        for i, v in enumerate(S):
            pos[v] = (0, y[i])
        for i, v in enumerate(N):
            pos[v] = (4, y[i])

        labels = {}
        
        # Signal vertices
        for idx, v in enumerate(S):
            j = idx // m + 1
            i = idx % m + 1
            labels[v] = rf"$S_{{{j},{i}}}$"
        
        # Noise vertices
        for idx, v in enumerate(N):
            j = idx // m + 1
            i = idx % m + 1
            labels[v] = rf"$N_{{{j},{i}}}$"

        #labels = {}    
        
        Sset = set(S)
        Nset = set(N)
        cut = []
        inside = []
        for u, v in G.edges():
            if (u in Sset and v in Nset) or (u in Nset and v in Sset):
                cut.append((u, v))
            else:
                inside.append((u, v))

        nx.draw_networkx_edges(G, pos, edgelist=inside, edge_color="0.75", width=1.5)
        nx.draw_networkx_edges(G, pos, edgelist=cut, edge_color="#1565C0", width=3)

        nx.draw_networkx_nodes(
            G, pos, nodelist=S, node_color="#43A047",
            edgecolors="black", linewidths=1.5, node_size=850,
        )
        nx.draw_networkx_nodes(
            G, pos, nodelist=N, node_color="#FB8C00",
            edgecolors="black", linewidths=1.5, node_size=850,
        )
        nx.draw_networkx_labels(G, pos, labels, font_size=14, font_weight="bold")

        plt.text(0, mk + 0.3, r"$S$", fontsize=18, ha="center")
        plt.text(4, mk + 0.3, r"$N$", fontsize=18, ha="center")

    plt.title(title, fontsize=18)
    plt.axis("off")
    plt.tight_layout()

    pdf_path = None
    if save:
        os.makedirs(fig_dir, exist_ok=True)
        filename = title
        for bad, good in [(" ", "_"), ("(", ""), (")", ""), ("/", "_"),
                           ("\\", "_"), (",", ""), ("=", ""), (":", "")]:
            filename = filename.replace(bad, good)
        pdf_path = os.path.join(fig_dir, filename + ".pdf")
        png_path = os.path.join(fig_dir, filename + ".png")
        plt.savefig(pdf_path, bbox_inches="tight")
        plt.savefig(png_path, dpi=600, bbox_inches="tight")

    plt.close()
    return pdf_path



In [6]:
# GSDC: decoder construction

def _coefficient_matrix(psi: np.ndarray, S: Sequence[int], N: Sequence[int], n: int, mk: int) -> np.ndarray:
    S = list(S)
    N = list(N)
    perm = S + N
    psi_t = np.transpose(psi.reshape([2] * n), perm)
    return psi_t.reshape(2 ** mk, 2 ** mk)


def construct_decoder(psi: np.ndarray, S: Sequence[int], N: Sequence[int], m: int, k: int, atol: float = 1e-9):
    mk = m * k
    n = len(S) + len(N)
    d = 2 ** mk
    M = _coefficient_matrix(psi, S, N, n, mk)
    Q = np.sqrt(d) * M
    W = Q.T

    unitarity_residual = np.linalg.norm(W.conj().T @ W - np.eye(d))

    Phi_mat = np.eye(d) / np.sqrt(d)
    reconstructed_M = Phi_mat @ W.T

    reconstruction_residual = np.linalg.norm(reconstructed_M - M)

    return {"W": W, "unitarity_residual": unitarity_residual, "is_unitary": unitarity_residual <= atol, "reconstruction_residual": reconstruction_residual, "reconstructs_state": reconstruction_residual <= atol,}


# Shared reporting routine

def report_certificate(label: str, A: np.ndarray, m: int, k: int, result: Optional[Tuple[Sequence[int], Sequence[int]]], build_state: bool = True, max_mk_for_state: int = 6, state_override: Optional[np.ndarray] = None,) -> None:
    
    mk = validate_graph_size(A, m, k)
    n = A.shape[0]

    print_parameters(m, k, A)
    print(f"Graph: {label}")

    if result is None:
        fig_path = plot_graph(A, title=label)
        print("  No valid balanced partition exists.")
        print("  Graph is NOT a valid encrypted cloning resource.")
        if fig_path:
            print(f"  Figure saved: {fig_path}")
        print()
        return

    S, N = result
    fig_path = plot_graph(A, S, N, title=label)
    print(f"  Certified! S = {set(S)}; N = {set(N)}")
    if fig_path:
        print(f"  Figure saved: {fig_path}")

    Gamma = cut_matrix(A, list(S), list(N))
    rank = gf2_rank(Gamma)
    print(f"  GF(2) rank(Gamma_S,N) = {rank}  (need {mk})  -> "
          f"{'PASS' if rank == mk else 'FAIL'}")

    if build_state and mk <= max_mk_for_state:
        psi = state_override if state_override is not None else graph_state_from_adjacency(A)
        rho_S = reduced_density_matrix(psi, list(S), n)
        matches, residual = rho_S_matches_maximally_mixed(rho_S, m, k)
        print(f"  rho_S == I/2^mk : {matches}   (||rho_S - I/2^mk||_F = {residual:.3e})")

        dec = construct_decoder(psi, S, N, m, k)
        print(f"  GSDC: W unitary : {dec['is_unitary']}   "
              f"(||W^dagger W - I||_F = {dec['unitarity_residual']:.3e})")
        print(f"  GSDC: |G> == (I_S ⊗ W)|Phi_{{{2 ** mk}}}> : "
              f"{dec['reconstructs_state']}   "
              f"(residual = {dec['reconstruction_residual']:.3e})")
    elif build_state:
        print(f"  (skipping explicit state construction: mk = {mk} > "
              f"max_mk_for_state = {max_mk_for_state}, state vector would "
              f"have 2^{n} = {2 ** n} amplitudes)")

    print()


def report_non_graph_state(label: str, psi: np.ndarray, n: int, m: int, k: int, S: Sequence[int], N: Sequence[int]) -> None:
    
    mk = m * k
    print_parameters(m, k, np.zeros((n, n), dtype=np.uint8))
    print(f"State: {label}  (S = {set(S)}, N = {set(N)})")
    rho_S = reduced_density_matrix(psi, list(S), n)
    matches, residual = rho_S_matches_maximally_mixed(rho_S, m, k)
    print(f"  rho_S == I/2^mk : {matches}   (||rho_S - I/2^mk||_F = {residual:.3e})  "
          f"-> {'valid encrypted-cloning resource' if matches else 'NOT a valid resource'}")

    dec = construct_decoder(psi, S, N, m, k)
    print(f"  GSDC: W unitary : {dec['is_unitary']}   "
          f"(||W^dagger W - I||_F = {dec['unitarity_residual']:.3e})")
    print()



In [7]:
print("=== GSECC certification + GSDC decoder construction ===\n")
print(f"Figures will be saved under: {FIGURE_DIR}\n")

# Common parameters for all examples
m, k = 3, 4

# -- Example 1: complete graph 
A = adjacency_matrix(2 * m * k, itertools.combinations(range(2 * m * k), 2))
result = find_certificate_exact(A, m, k)
report_certificate("Complete graph", A, m, k, result)

# -- Example 2: cycle graph 
A = adjacency_matrix(2 * m * k, [(i, (i + 1) % (2 * m * k)) for i in range(2 * m * k)])

result = find_certificate_exact(A, m, k)
report_certificate("Cycle graph", A, m, k, result)

# -- Example 3: linear cluster chain 
A = path_graph(m, k)
result = find_certificate_exact(A, m, k)
report_certificate("Path / linear cluster chain", A, m, k, result)

# -- Example 4: 2x4 cluster grid 
rows, cols = 2, 2 * m * k // 2   # choose dimensions with rows*cols = 2mk
A = grid_graph(rows, cols)
result = find_certificate_exact(A, m, k)
report_certificate(f"{rows}x{cols} cluster grid", A, m, k, result)

# -- Example 5: dense Erdos-Renyi random graph 
n = 2 * m * k
p = 0.9
rng = random.Random(42)
edges = [(i, j) for i in range(n)
         for j in range(i + 1, n)
         if rng.random() < p]
A = adjacency_matrix(n, edges)
result = find_certificate_exact(A, m, k)
report_certificate(
    f"G(n={n}, p={p}) dense random graph",
    A, m, k, result,
    build_state=False
)

# -- Example 6: sparse Erdos-Renyi random graph 
n = 2 * m * k
p = 0.35
rng = random.Random(7)
edges = [(i, j) for i in range(n)
         for j in range(i + 1, n)
         if rng.random() < p]
A = adjacency_matrix(n, edges)
result = find_certificate_exact(A, m, k)
report_certificate(
    f"G(n={n}, p={p}) sparse random graph",
    A, m, k, result,
    build_state=True,
    max_mk_for_state=6
)

# -- Example 7: Bell-pair matching graph 
A = bell_pair_matching_graph(m, k)
result = find_certificate_exact(A, m, k)
report_certificate(f"Bell-pair matching graph ({m * k} Bell pairs)", A, m, k, result)

# -- Example 8: GHZ / star graph 
A = star_graph(m, k)
result = find_certificate_exact(A, m, k)
report_certificate("GHZ (star graph)", A, m, k, result)

# -- Example 8b: actual GHZ state vector 
n = 2 * m * k
psi_ghz = ghz_state(n)
mk = m * k
S_arbitrary = tuple(range(mk))
N_arbitrary = tuple(range(mk, n))
report_non_graph_state("GHZ state vector (arbitrary balanced cut)", psi_ghz, n, m, k, S_arbitrary, N_arbitrary)

# -- Example 9: mismatched (m, k) 
print("Deliberately mismatched (m, k) example:")
try:
    A_bad = adjacency_matrix(
        10,
        itertools.combinations(range(10), 2)
    )  # intentionally wrong size
    find_certificate_exact(A_bad, m, k)
except ValueError as e:
    print(f"  Rejected as expected: {e}")
print()

if os.path.isdir(FIGURE_DIR):
    saved = sorted(f for f in os.listdir(FIGURE_DIR) if f.endswith(".pdf"))
    print(f"Saved {len(saved)} figures (PDF + 600 dpi PNG each) to {FIGURE_DIR}:")
    for f in saved:
        print(f"  {f}")

=== GSECC certification + GSDC decoder construction ===

Figures will be saved under: C:\Users\roypr\OneDrive\Documents\1.Project files\Quantum cloning\Encrypted clonning\final_plotting

Encrypted Cloning Parameters
----------------------------
m  = 3
k  = 4
mk = 12
Graph vertices = 24
Graph: Complete graph
  No valid balanced partition exists.
  Graph is NOT a valid encrypted cloning resource.
  Figure saved: C:\Users\roypr\OneDrive\Documents\1.Project files\Quantum cloning\Encrypted clonning\final_plotting\Complete_graph.pdf



findfont: Font family ['cmsy10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmr10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmtt10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmmi10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmb10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmss10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmex10'] not found. Falling back to DejaVu Sans.


Encrypted Cloning Parameters
----------------------------
m  = 3
k  = 4
mk = 12
Graph vertices = 24
Graph: Cycle graph
  Certified! S = {0, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21}; N = {2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 23}
  Figure saved: C:\Users\roypr\OneDrive\Documents\1.Project files\Quantum cloning\Encrypted clonning\final_plotting\Cycle_graph.pdf
  GF(2) rank(Gamma_S,N) = 12  (need 12)  -> PASS
  (skipping explicit state construction: mk = 12 > max_mk_for_state = 6, state vector would have 2^24 = 16777216 amplitudes)

Encrypted Cloning Parameters
----------------------------
m  = 3
k  = 4
mk = 12
Graph vertices = 24
Graph: Path / linear cluster chain
  Certified! S = {0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22}; N = {1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23}
  Figure saved: C:\Users\roypr\OneDrive\Documents\1.Project files\Quantum cloning\Encrypted clonning\final_plotting\Path___linear_cluster_chain.pdf
  GF(2) rank(Gamma_S,N) = 12  (need 12)  -> PASS
  (skipping explicit s